In [ ]:
!pip install albumentations

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gdown
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from PIL import Image
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
import cv2

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Download
file_id = '1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

# 2. Unzip
if os.path.exists(output) and not os.path.exists("/content/data/data"):
    print("Unzipping...")
    !unzip -o -q data.zip -d /content/data
    print("Data ready at /content/data/data/")
else:
    print("Data already extracted.")

IMG_PATH = "/content/data/data/images" 
MASK_PATH = "/content/data/data/masks"

In [ ]:
# --- 1. Augmentations ---
train_transform = A.Compose([
    A.Resize(height=256, width=256, interpolation=cv2.INTER_LINEAR),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3, border_mode=cv2.BORDER_CONSTANT),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

val_transform = A.Compose([
    A.Resize(height=256, width=256, interpolation=cv2.INTER_LINEAR),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

# --- 2. Dataset Class ---
class CustomDataset(Dataset):
    def __init__(self, image_dir, mask_dir, file_list, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = file_list
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            mask = mask.unsqueeze(0).float()
            mask = (mask > 0.5).float()

        return image, mask

In [ ]:
# --- Basic Conv Block ---
class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size, stride, padding, dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return self.relu(x)

# --- Receptive Field Block (RFB) ---
class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(RFB_modified, self).__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(BasicConv2d(in_channel, out_channel, 1))
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4*out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), 1))
        x = self.relu(x_cat + self.conv_res(x))
        return x

# --- Partial Decoder (Aggregation) - FIXED ---
class aggregation(nn.Module):
    def __init__(self, channel):
        super(aggregation, self).__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        
        self.conv_concat2 = BasicConv2d(2*channel, 2*channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2*channel, 2*channel, 3, padding=1) # FIX
        self.conv_concat3 = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        
        self.conv4 = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3*channel, 1, 1)

    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * \
               self.conv_upsample3(self.upsample(x2)) * x3

        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)

        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)

        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x

# --- PraNet Main Class ---
class PraNet(nn.Module):
    def __init__(self, channel=32):
        super(PraNet, self).__init__()
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        self.rfb2_1 = RFB_modified(512, channel)
        self.rfb3_1 = RFB_modified(1024, channel)
        self.rfb4_1 = RFB_modified(2048, channel)
        
        self.agg1 = aggregation(channel)
        
        self.ra4_conv1 = BasicConv2d(2048, 256, kernel_size=1)
        self.ra4_conv2 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv3 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv4 = BasicConv2d(256, 1, kernel_size=1)
        
        self.ra3_conv1 = BasicConv2d(1024, 64, kernel_size=1)
        self.ra3_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv4 = BasicConv2d(64, 1, kernel_size=1)
        
        self.ra2_conv1 = BasicConv2d(512, 64, kernel_size=1)
        self.ra2_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv4 = BasicConv2d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x) 

        x1 = self.resnet.layer1(x)
        x2 = self.resnet.layer2(x1)
        x3 = self.resnet.layer3(x2)
        x4 = self.resnet.layer4(x3)

        x2_rfb = self.rfb2_1(x2)
        x3_rfb = self.rfb3_1(x3)
        x4_rfb = self.rfb4_1(x4)

        ra5_feat = self.agg1(x4_rfb, x3_rfb, x2_rfb)
        lateral_map_5 = F.interpolate(ra5_feat, scale_factor=8, mode='bilinear', align_corners=True)

        crop_4 = F.interpolate(ra5_feat, scale_factor=0.25, mode='bilinear', align_corners=True)
        x = -1*(torch.sigmoid(crop_4)) + 1
        x = x.expand(-1, 2048, -1, -1).mul(x4)
        x = self.ra4_conv1(x)
        x = F.relu(self.ra4_conv2(x))
        x = F.relu(self.ra4_conv3(x))
        ra4_feat = self.ra4_conv4(x)
        x = ra4_feat + crop_4
        lateral_map_4 = F.interpolate(x, scale_factor=32, mode='bilinear', align_corners=True)

        crop_3 = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = -1*(torch.sigmoid(crop_3)) + 1
        x = x.expand(-1, 1024, -1, -1).mul(x3)
        x = self.ra3_conv1(x)
        x = F.relu(self.ra3_conv2(x))
        x = F.relu(self.ra3_conv3(x))
        ra3_feat = self.ra3_conv4(x)
        x = ra3_feat + crop_3
        lateral_map_3 = F.interpolate(x, scale_factor=16, mode='bilinear', align_corners=True)

        crop_2 = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = -1*(torch.sigmoid(crop_2)) + 1
        x = x.expand(-1, 512, -1, -1).mul(x2)
        x = self.ra2_conv1(x)
        x = F.relu(self.ra2_conv2(x))
        x = F.relu(self.ra2_conv3(x))
        ra2_feat = self.ra2_conv4(x)
        x = ra2_feat + crop_2
        lateral_map_2 = F.interpolate(x, scale_factor=8, mode='bilinear', align_corners=True)

        return lateral_map_5, lateral_map_4, lateral_map_3, lateral_map_2

In [ ]:
# --- Structure Loss ---
def structure_loss(pred, mask):
    bce = F.binary_cross_entropy_with_logits(pred, mask)
    pred = torch.sigmoid(pred)
    intersection = (pred * mask).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + mask.sum(dim=(2, 3))
    dice = 1 - (2. * intersection + 1e-5) / (union + 1e-5)
    return bce + dice.mean()

# --- Experiment Runner ---
def run_experiment(seed, epochs=20):
    print(f"\n{'='*40}")
    print(f"STARTING RUN WITH SEED: {seed}")
    print(f"{'='*40}")
    
    # 1. Split Data
    all_files = sorted(os.listdir(IMG_PATH))
    train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=seed)
    
    train_dataset = CustomDataset(IMG_PATH, MASK_PATH, train_files, transform=train_transform)
    test_dataset = CustomDataset(IMG_PATH, MASK_PATH, test_files, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # 2. Initialize Model
    model = PraNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # 3. Training Loop
    print(f"Training for {epochs} epochs...")
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            
            # Forward (PraNet outputs 4 maps)
            out5, out4, out3, out2 = model(images)
            
            # Deep Supervision Loss
            loss5 = structure_loss(out5, masks)
            loss4 = structure_loss(out4, masks)
            loss3 = structure_loss(out3, masks)
            loss2 = structure_loss(out2, masks)
            
            loss = loss5 + loss4 + loss3 + loss2
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")
            
    return model, test_loader

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    scores = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}
    
    with torch.no_grad():
        for image, true_mask in test_loader:
            input_batch = image.to(device)
            
            # --- Forward (Ignore first 3 maps) ---
            _, _, _, pred_map = model(input_batch)
            
            # Sigmoid & Threshold
            probs = pred_map.sigmoid()
            pred_mask = (probs > 0.5).float()
            
            p = pred_mask.cpu().squeeze().numpy().flatten().astype(np.uint8)
            g = true_mask.cpu().squeeze().numpy().flatten().astype(np.uint8)
            
            scores["dice"].append(f1_score(g, p, zero_division=1))
            scores["iou"].append(jaccard_score(g, p, zero_division=1))
            scores["prec"].append(precision_score(g, p, zero_division=1))
            scores["rec"].append(recall_score(g, p, zero_division=1))
            scores["acc"].append(accuracy_score(g, p))
            
    return {k: np.mean(v) for k, v in scores.items()}

In [ ]:
SEEDS = [42, 0, 123, 7, 2024]
EPOCHS_PER_RUN = 50 # PraNet converges slower than UNet, 50-100 recommended

all_results = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}

for seed in SEEDS:
    # 1. Train
    model, test_loader = run_experiment(seed, epochs=EPOCHS_PER_RUN)
    
    # 2. Evaluate
    metrics = evaluate_model(model, test_loader)
    
    # 3. Store
    for k, v in metrics.items():
        all_results[k].append(v)
        
    print(f"Run {seed} -> Dice: {metrics['dice']:.4f} | IoU: {metrics['iou']:.4f}")

# --- Report ---
print("\n" + "="*50)
print("FINAL MULTI-RUN REPORT (PraNet)")
print("="*50)
print(f"{'Metric':<10} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 35)

for k, v in all_results.items():
    print(f"{k.upper():<10} | {np.mean(v):.4f}     | {np.std(v):.4f}")

In [ ]:
# --- Configuration ---
ITERATIONS = 100
WARMUP = 50 

print(f"⏱️ Benchmarking Inference Speed (PraNet) on {device}...")
model.eval()

# 1. Warmup
dummy_input = torch.randn(1, 3, 256, 256).to(device)
print(f"   Warming up for {WARMUP} steps...")
with torch.no_grad():
    for _ in range(WARMUP):
        _ = model(dummy_input)

# 2. Measurement
timings = []
print(f"   Measuring over {ITERATIONS} runs...")

with torch.no_grad():
    for _ in range(ITERATIONS):
        if device.type == 'cuda':
            torch.cuda.synchronize()
            start = time.time()
            _ = model(dummy_input)
            torch.cuda.synchronize()
            end = time.time()
        else:
            start = time.time()
            _ = model(dummy_input)
            end = time.time()
        
        timings.append((end - start) * 1000) # ms

avg_ms = np.mean(timings)
std_ms = np.std(timings)
fps = 1000 / avg_ms

print("\n" + "="*30)
print("🚀 FINAL SPEED RESULTS")
print("="*30)
print(f"Avg Time: {avg_ms:.2f} ms ± {std_ms:.2f} ms")
print(f"FPS:      {fps:.2f} frames/sec")
print("="*30)